
# Classical Machine Learning — From Scratch Exam Review

### هدف این نوت‌بوک
این نوت‌بوک برای مرور **سریع و عملی قبل از آزمون** طراحی شده است.

قانون اصلی:
> اول شهود → بعد فرمول → بعد ساده‌ترین کد ممکن → بعد یک plot → بعد یک نسخه‌ی خیلی کوتاه از API کتابخانه‌ای.

در این نوت‌بوک عمداً از implementationهای سنگین و frameworkهای پیچیده دوری می‌کنیم. هدف این است که بتوانی در جلسه، اسکلت الگوریتم را **بدون AI بازسازی کنی**.

## نقشه

1. Regression
2. Classification
3. Trees
4. Ensembles
5. Unsupervised Learning
6. Metrics & Preprocessing
7. Final Cheat Sheet

> بیشتر الگوریتم‌ها با NumPy از صفر نوشته می‌شوند. در انتهای هر بخش، نسخه‌ی scikit-learn را فقط برای عادت‌کردن به API می‌بینیم.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

print("NumPy:", np.__version__)



# 0. الگوی مشترک بیشتر الگوریتم‌ها

در آزمون، این چرخه را زیاد می‌بینی:

```text
Initialize
   ↓
Predict / Assign
   ↓
Calculate Error / Score
   ↓
Update
   ↓
Repeat
```

مثلاً:

- Linear Regression → update با gradient
- Logistic Regression → update با gradient
- K-Means → assignment سپس mean
- Decision Tree → best split
- Gradient Boosting → residual



# 1. Regression

## 1.1 Linear Regression

هدف:

\[
\hat y = Xw + b
\]

برای MSE:

\[
L = \frac{1}{n}\sum_{i=1}^{n}(\hat y_i-y_i)^2
\]

گرادیان نسبت به وزن‌ها:

\[
\nabla_w L = \frac{2}{n}X^T(\hat y-y)
\]

و برای bias:

\[
\nabla_b L = \frac{2}{n}\sum_i(\hat y_i-y_i)
\]

### چیزی که باید حفظ کنی

```text
prediction = X @ w + b
error      = prediction - y
dw         = (2/n) * X.T @ error
db         = (2/n) * sum(error)
parameter -= lr * gradient
```


In [ ]:

# Simple 1D regression data
X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
y = np.array([3.1, 5.0, 7.2, 8.9, 11.1])

def linear_regression_fit(X, y, lr=0.01, epochs=2000):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0

    for _ in range(epochs):
        y_pred = X @ w + b
        error = y_pred - y

        dw = (2 / n) * X.T @ error
        db = (2 / n) * np.sum(error)

        w -= lr * dw
        b -= lr * db

    return w, b

def linear_predict(X, w, b):
    return X @ w + b

w, b = linear_regression_fit(X, y, lr=0.01, epochs=2000)
pred = linear_predict(X, w, b)

print("w =", w)
print("b =", b)
print("predictions =", np.round(pred, 2))


In [ ]:

x_plot = np.linspace(0.5, 5.5, 100).reshape(-1, 1)

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], y, label="data")
plt.plot(x_plot[:, 0], linear_predict(x_plot, w, b), label="fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression from Scratch")
plt.legend()
plt.show()



## 1.2 Polynomial Regression

نکته‌ی بسیار مهم:

> Polynomial Regression یک الگوریتم کاملاً جدید نیست؛  
> **feature engineering + Linear Regression** است.

برای درجه‌ی 3:

\[
x \rightarrow [x,x^2,x^3]
\]

سپس دوباره:

\[
\hat y = X_{poly}w+b
\]


In [ ]:

def polynomial_features(X, degree):
    features = [X ** i for i in range(1, degree + 1)]
    return np.concatenate(features, axis=1)

X_curve = np.linspace(-2, 2, 30).reshape(-1, 1)
y_curve = X_curve[:, 0]**2 + 0.3*np.random.randn(30)

X_poly = polynomial_features(X_curve, degree=2)
w_poly, b_poly = linear_regression_fit(X_poly, y_curve, lr=0.01, epochs=5000)

x_dense = np.linspace(-2, 2, 200).reshape(-1, 1)
y_dense = linear_predict(polynomial_features(x_dense, 2), w_poly, b_poly)

plt.figure(figsize=(6, 4))
plt.scatter(X_curve[:, 0], y_curve, s=25)
plt.plot(x_dense[:, 0], y_dense)
plt.title("Polynomial Regression = Feature Expansion + Linear Regression")
plt.show()



## 1.3 Ridge Regression

Ridge یعنی:

\[
L = MSE + \lambda \|w\|_2^2
\]

در نتیجه:

\[
\nabla_w L
=
\frac{2}{n}X^T(\hat y-y)+2\lambda w
\]

### حفظی

\[
\boxed{\text{Ridge = MSE + L2}}
\]

L2 وزن‌ها را کوچک می‌کند.


In [ ]:

def ridge_fit(X, y, lr=0.01, epochs=2000, lam=0.1):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0

    for _ in range(epochs):
        y_pred = X @ w + b
        error = y_pred - y

        dw = (2/n) * X.T @ error + 2*lam*w
        db = (2/n) * np.sum(error)

        w -= lr * dw
        b -= lr * db

    return w, b

w_ridge, b_ridge = ridge_fit(X, y, lam=0.2)
print("Ridge weights:", w_ridge, "bias:", b_ridge)



## 1.4 Lasso Regression

Lasso:

\[
L = MSE+\lambda\sum_j |w_j|
\]

مشتق/زیرگرادیان L1:

\[
\frac{\partial |w_j|}{\partial w_j}\approx \operatorname{sign}(w_j)
\]

### حفظی

\[
\boxed{\text{Lasso = MSE + L1}}
\]

اثر مشهور L1:
> بعضی ضرایب می‌توانند دقیقاً به صفر برسند → feature selection.


In [ ]:

def lasso_fit(X, y, lr=0.01, epochs=2000, lam=0.1):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0

    for _ in range(epochs):
        y_pred = X @ w + b
        error = y_pred - y

        dw = (2/n) * X.T @ error + lam*np.sign(w)
        db = (2/n) * np.sum(error)

        w -= lr * dw
        b -= lr * db

    return w, b

w_lasso, b_lasso = lasso_fit(X_poly, y_curve, lam=0.01)
print("Lasso weights:", w_lasso, "bias:", b_lasso)



# 2. Classification

## 2.1 Logistic Regression

ابتدا یک score خطی:

\[
z=Xw+b
\]

سپس sigmoid:

\[
\sigma(z)=\frac{1}{1+e^{-z}}
\]

که یک probability تقریبی می‌دهد:

\[
p(y=1|x)=\sigma(z)
\]

Binary Cross-Entropy:

\[
L=-\frac1n\sum_i
\left[
y_i\log p_i+(1-y_i)\log(1-p_i)
\right]
\]

نکته‌ی طلایی:

\[
\boxed{
\nabla_w L=\frac1n X^T(p-y)
}
\]

یعنی implementation بسیار شبیه Linear Regression است؛ فقط prediction عوض شده است.


In [ ]:

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_fit(X, y, lr=0.1, epochs=1000):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0

    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        error = p - y

        dw = (1/n) * X.T @ error
        db = (1/n) * np.sum(error)

        w -= lr * dw
        b -= lr * db

    return w, b

def logistic_predict(X, w, b):
    p = sigmoid(X @ w + b)
    return (p >= 0.5).astype(int)

X_cls = np.array([
    [1, 1], [1.2, 1.1], [0.8, 1.3],
    [4, 4], [4.2, 3.8], [3.7, 4.1]
], dtype=float)
y_cls = np.array([0, 0, 0, 1, 1, 1])

w_log, b_log = logistic_fit(X_cls, y_cls)
print("pred:", logistic_predict(X_cls, w_log, b_log))


In [ ]:

plt.figure(figsize=(6, 4))
plt.scatter(X_cls[:, 0], X_cls[:, 1], c=y_cls, s=60)

x1 = np.linspace(0, 5, 100)
x2 = -(w_log[0]*x1 + b_log) / (w_log[1] + 1e-12)

plt.plot(x1, x2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Logistic Regression Decision Boundary")
plt.show()



## 2.2 Perceptron

Perceptron تقریباً ساده‌ترین classifier خطی است:

\[
\hat y=
\begin{cases}
1 & w^Tx+b\ge0\\
0 & \text{otherwise}
\end{cases}
\]

اگر اشتباه کرد:

\[
w \leftarrow w+\eta(y-\hat y)x
\]

### فرق با Logistic

```text
Perceptron → hard 0/1 decision
Logistic   → probability + sigmoid
```


In [ ]:

def perceptron_fit(X, y, lr=0.1, epochs=20):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0

    for _ in range(epochs):
        for i in range(n):
            score = X[i] @ w + b
            pred = 1 if score >= 0 else 0

            update = lr * (y[i] - pred)

            w += update * X[i]
            b += update

    return w, b

w_p, b_p = perceptron_fit(X_cls, y_cls)
print("Perceptron pred:", logistic_predict(X_cls, w_p, b_p))



## 2.3 KNN

KNN تقریباً هیچ trainingی ندارد.

برای یک sample جدید:

1. فاصله تا همه‌ی trainها
2. مرتب‌سازی
3. انتخاب k همسایه
4. majority vote

برای Euclidean:

\[
d(x,x_i)=\sqrt{\sum_j(x_j-x_{ij})^2}
\]


In [ ]:

def knn_predict(X_train, y_train, x, k=3):
    distances = np.sqrt(np.sum((X_train - x)**2, axis=1))
    nearest_idx = np.argsort(distances)[:k]
    labels = y_train[nearest_idx]

    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]

for x in [[1.1, 1.0], [4.0, 4.2]]:
    print(x, "->", knn_predict(X_cls, y_cls, np.array(x), k=3))



## 2.4 Gaussian Naive Bayes

Bayes:

\[
P(y|x)\propto P(x|y)P(y)
\]

فرض Naive:

\[
P(x|y)=\prod_jP(x_j|y)
\]

برای feature پیوسته، هر feature را Gaussian در نظر می‌گیریم:

\[
p(x_j|y=c)
=
\frac{1}{\sqrt{2\pi\sigma_{jc}^2}}
\exp
\left(
-\frac{(x_j-\mu_{jc})^2}{2\sigma_{jc}^2}
\right)
\]

### حفظی

```text
برای هر کلاس:
mean + variance + prior
↓
likelihood
↓
product
↓
بیشترین posterior
```


In [ ]:

def gaussian_pdf(x, mean, var):
    return np.exp(-(x-mean)**2/(2*var)) / np.sqrt(2*np.pi*var)

def gnb_fit(X, y):
    classes = np.unique(y)
    means, vars_, priors = {}, {}, {}

    for c in classes:
        Xc = X[y == c]
        means[c] = Xc.mean(axis=0)
        vars_[c] = Xc.var(axis=0) + 1e-9
        priors[c] = len(Xc) / len(X)

    return classes, means, vars_, priors

def gnb_predict(x, model):
    classes, means, vars_, priors = model
    scores = {}

    for c in classes:
        likelihoods = gaussian_pdf(x, means[c], vars_[c])
        scores[c] = np.prod(likelihoods) * priors[c]

    return max(scores, key=scores.get)

gnb_model = gnb_fit(X_cls, y_cls)
print(gnb_predict(np.array([1.0, 1.2]), gnb_model))
print(gnb_predict(np.array([4.0, 4.0]), gnb_model))



## 2.5 SVM

SVM دنبال boundaryای است که **margin** را زیاد کند.

برای labelهای \(y\in\{-1,+1\}\):

\[
L_i=\max(0,1-y_i(w^Tx_i+b))
\]

به علاوه regularization:

\[
L=
\frac12\|w\|^2+C\sum_i L_i
\]

نکته:

```text
margin >= 1 → hinge loss صفر
margin < 1  → hinge loss فعال
```


In [ ]:

def svm_fit(X, y, lr=0.001, epochs=1000, C=1.0):
    y2 = np.where(y == 0, -1, 1)
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0

    for _ in range(epochs):
        for i in range(n):
            margin = y2[i] * (X[i] @ w + b)

            if margin >= 1:
                dw = w
                db = 0.0
            else:
                dw = w - C*y2[i]*X[i]
                db = -C*y2[i]

            w -= lr * dw
            b -= lr * db

    return w, b

w_svm, b_svm = svm_fit(X_cls, y_cls, lr=0.001, epochs=2000)
print("SVM done.")



# 3. Decision Tree

Decision Tree با gradient کار نمی‌کند.

برای classification سؤال اصلی:

> کدام feature و threshold داده را بهتر split می‌کنند؟

یکی از معیارها:

\[
Gini=1-\sum_k p_k^2
\]

برای split:

\[
Gini_{split}
=
\frac{n_L}{n}Gini_L+
\frac{n_R}{n}Gini_R
\]

و بهترین split = کمترین score.

### الگوی ذهنی

```text
best split
   ↓
left child + right child
   ↓
best split برای هر child
   ↓
repeat تا leaf
```


In [ ]:

def gini(y):
    if len(y) == 0:
        return 0.0

    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)

    return 1 - np.sum(p**2)

def best_split(X, y):
    best_feature = None
    best_threshold = None
    best_score = float("inf")

    n, d = X.shape

    for j in range(d):
        for t in np.unique(X[:, j]):
            left = y[X[:, j] <= t]
            right = y[X[:, j] > t]

            if len(left) == 0 or len(right) == 0:
                continue

            score = (len(left)/n)*gini(left) + (len(right)/n)*gini(right)

            if score < best_score:
                best_score = score
                best_feature = j
                best_threshold = t

    return best_feature, best_threshold, best_score

print("Best split:", best_split(X_cls, y_cls))



> برای مرور آزمون، هسته‌ی Decision Tree یعنی **best split** مهم‌تر از پیاده‌سازی recursive کامل است. درخت کامل فقط همین فرایند را دوباره و دوباره روی childها اجرا می‌کند.



# 4. Ensembles

## 4.1 Bagging

ایده:

- چند bootstrap sample می‌سازیم.
- مدل‌ها مستقل از هم train می‌شوند.
- در پایان vote/average می‌کنیم.

```text
Dataset
 ├─ bootstrap 1 → model 1
 ├─ bootstrap 2 → model 2
 ├─ bootstrap 3 → model 3
 └─ ...
             ↓
        aggregate
```

\[
\hat y = mode(f_1(x),...,f_B(x))
\]

### جمله‌ی حفظی

\[
\boxed{\text{Bagging = Bootstrap + Parallel + Aggregate}}
\]


In [ ]:

def bootstrap_sample(X, y):
    n = len(X)
    idx = np.random.choice(n, size=n, replace=True)
    return X[idx], y[idx]

Xb, yb = bootstrap_sample(X_cls, y_cls)
print("bootstrap X:", Xb)
print("bootstrap y:", yb)



## 4.2 Random Forest

Random Forest تقریباً:

\[
\boxed{\text{Random Forest = Bagging + Random Feature Selection + Trees}}
\]

پس:

```text
Bagging:
random samples

Random Forest:
random samples
+
random subset of features
+
decision trees
```



## 4.3 AdaBoost

اینجا مدل‌ها پشت سر هم ساخته می‌شوند.

```text
model 1
   ↓
نمونه‌های اشتباه مهم‌تر می‌شوند
   ↓
model 2
   ↓
اشتباه‌های باقی‌مانده مهم‌تر
   ↓
model 3
```

ابتدا وزن نمونه‌ها:

\[
w_i=\frac1n
\]

خطای weighted:

\[
\epsilon=
\frac{\sum_iw_i\,1[y_i\neq h(x_i)]}
{\sum_iw_i}
\]

ضریب مدل:

\[
\alpha=
\frac12\ln\frac{1-\epsilon}{\epsilon}
\]

### چیزی که باید یادت بماند

> AdaBoost با **وزن‌دهی به نمونه‌های سخت/اشتباه** مدل بعدی را روی نقاط مشکل‌دار متمرکز می‌کند.


In [ ]:

# Core AdaBoost update (binary labels -1/+1)
def adaboost_weight_update(weights, y, pred):
    y2 = np.where(y == 0, -1, 1)
    p2 = np.where(pred == 0, -1, 1)

    error = np.sum(weights * (y2 != p2))
    alpha = 0.5 * np.log((1 - error) / (error + 1e-12))

    weights = weights * np.exp(-alpha * y2 * p2)
    weights = weights / np.sum(weights)

    return weights, error, alpha

weights = np.ones(len(y_cls)) / len(y_cls)
pred_example = np.array([0, 1, 0, 1, 0, 1])

weights, error, alpha = adaboost_weight_update(weights, y_cls, pred_example)
print("error =", error)
print("alpha =", alpha)
print("new weights =", np.round(weights, 3))



## 4.4 Gradient Boosting

مدل‌ها پشت سر هم می‌آیند، ولی این بار ایده‌ی اصلی:

> مدل بعدی **residual / negative gradient** را یاد بگیرد.

برای MSE:

\[
r_i=y_i-F(x_i)
\]

و:

\[
F_m(x)=F_{m-1}(x)+\eta h_m(x)
\]

### الگوی ذهنی

```text
prediction فعلی
      ↓
residual = y - prediction
      ↓
tree جدید residual را یاد می‌گیرد
      ↓
prediction += learning_rate * update
      ↓
repeat
```


In [ ]:

# Minimal Gradient Boosting core using a simple stump-like threshold search.
# This is intentionally compact: the goal is to remember the mechanism.

def fit_stump_regression(X, r):
    # One feature, one threshold: predict mean on each side.
    x = X[:, 0]
    best = None
    best_mse = float("inf")

    for t in np.unique(x):
        left = x <= t
        right = ~left
        if left.sum() == 0 or right.sum() == 0:
            continue

        left_value = r[left].mean()
        right_value = r[right].mean()

        pred = np.where(left, left_value, right_value)
        mse = np.mean((r - pred)**2)

        if mse < best_mse:
            best_mse = mse
            best = (t, left_value, right_value)

    return best

def predict_stump(X, stump):
    t, left_value, right_value = stump
    return np.where(X[:, 0] <= t, left_value, right_value)

X_gb = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])
y_gb = np.array([1.0, 1.5, 3.0, 3.2, 5.0, 5.5])

base = np.mean(y_gb)
pred = np.full(len(y_gb), base)
stumps = []

for _ in range(5):
    residual = y_gb - pred
    stump = fit_stump_regression(X_gb, residual)
    update = predict_stump(X_gb, stump)

    pred += 0.3 * update
    stumps.append(stump)

print("true:", y_gb)
print("pred:", np.round(pred, 2))


In [ ]:

plt.figure(figsize=(6, 4))
plt.scatter(X_gb[:, 0], y_gb, label="data")
plt.plot(X_gb[:, 0], pred, marker="o", label="gradient boosting")
plt.legend()
plt.title("Gradient Boosting learns residuals")
plt.show()



# 5. Unsupervised Learning

## 5.1 K-Means

سه کلمه‌ی اصلی:

\[
\boxed{Assign \rightarrow Mean \rightarrow Repeat}
\]

هدف:

\[
\min_{\mu_1,\dots,\mu_k}
\sum_i \|x_i-\mu_{c_i}\|^2
\]

در هر iteration:

1. هر point را به نزدیک‌ترین center بده.
2. center هر cluster را mean کن.
3. تکرار.



In [ ]:

def kmeans(X, k, epochs=100):
    idx = np.random.choice(len(X), k, replace=False)
    centers = X[idx].copy()

    for _ in range(epochs):
        distances = np.sqrt(
            ((X[:, None] - centers[None, :])**2).sum(axis=2)
        )

        labels = np.argmin(distances, axis=1)

        new_centers = np.array([
            X[labels == j].mean(axis=0)
            if np.any(labels == j) else centers[j]
            for j in range(k)
        ])

        if np.allclose(centers, new_centers):
            break

        centers = new_centers

    return centers, labels

X_km = np.vstack([
    np.random.randn(40, 2) + np.array([0, 0]),
    np.random.randn(40, 2) + np.array([4, 4]),
    np.random.randn(40, 2) + np.array([0, 5]),
])

centers, labels = kmeans(X_km, k=3)

plt.figure(figsize=(6, 4))
plt.scatter(X_km[:, 0], X_km[:, 1], c=labels, s=20)
plt.scatter(centers[:, 0], centers[:, 1], marker="X", s=150)
plt.title("K-Means from Scratch")
plt.show()



## 5.2 PCA

PCA دنبال directionهایی است که **بیشترین variance** را نگه دارند.

مراحل:

```text
X
↓
center: X - mean
↓
covariance
↓
eigenvectors
↓
بزرگ‌ترین eigenvalues
↓
project
```

\[
C=\frac1nX_c^TX_c
\]

اگر \(v\) eigenvector باشد:

\[
Cv=\lambda v
\]

بزرگ‌ترین \(\lambda\) یعنی آن direction بیشترین variance را توضیح می‌دهد.

### نسخه‌ی خیلی ساده با SVD

\[
X_c=U\Sigma V^T
\]

و componentهای اصلی از سطرهای \(V^T\) می‌آیند.


In [ ]:

def pca_svd(X, n_components=2):
    Xc = X - X.mean(axis=0)

    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

    components = Vt[:n_components]
    X_new = Xc @ components.T

    return X_new, components, S

X_pca = np.hstack([
    np.random.randn(100, 1),
    2*np.random.randn(100, 1) + np.random.randn(100, 1)*0.2
])

X_new, components, S = pca_svd(X_pca, n_components=1)

plt.figure(figsize=(6, 4))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=20)
plt.title("Original data before PCA")
plt.show()

print("Principal component:", components)



## 5.3 GMM

Gaussian Mixture Model را این‌طور ببین:

> K-Means یک **hard assignment** می‌دهد، GMM یک **soft assignment**.

K-Means:

```text
point → cluster 2
```

GMM:

```text
point → 0.1 cluster 1
        0.7 cluster 2
        0.2 cluster 3
```

الگوریتم EM:

```text
Initialize
   ↓
E-step: responsibility
   ↓
M-step: update μ, Σ, π
   ↓
Repeat
```

\[
\gamma_{ik}
=
\frac{\pi_k\mathcal N(x_i|\mu_k,\Sigma_k)}
{\sum_j\pi_j\mathcal N(x_i|\mu_j,\Sigma_j)}
\]

در آزمون، **E-step/M-step و مفهوم responsibility** را سریع به یاد بیاور؛ پیاده‌سازی کامل GMM معمولاً از بقیه مهم‌تر نیست.



# 6. Metrics & Preprocessing

## Classification

Confusion Matrix:

```text
                predicted
              0         1
actual 0      TN        FP
actual 1      FN        TP
```

\[
Precision=\frac{TP}{TP+FP}
\]

\[
Recall=\frac{TP}{TP+FN}
\]

\[
F1=2\frac{Precision\cdot Recall}
{Precision+Recall}
\]

Accuracy:

\[
\frac{TP+TN}{TP+TN+FP+FN}
\]

### نکته
وقتی classها نامتوازن هستند، فقط Accuracy کافی نیست؛ Precision/Recall/F1 و metricهای متناسب با مسئله را بررسی کن.



## Regression Metrics

\[
MSE=\frac1n\sum_i(y_i-\hat y_i)^2
\]

\[
RMSE=\sqrt{MSE}
\]

\[
MAE=\frac1n\sum_i|y_i-\hat y_i|
\]

\[
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}
{\sum_i(y_i-\bar y)^2}
\]



## Scaling

### Standardization

\[
x'=\frac{x-\mu}{\sigma}
\]

### Min-Max

\[
x'=\frac{x-x_{min}}{x_{max}-x_{min}}
\]

نکته‌ی آزمونی:

> scaler را روی **train fit** کن و روی validation/test فقط transform کن.



# 7. اسکلت API در scikit-learn

هدف این قسمت حفظ تمام APIها نیست؛ فقط به این الگو عادت کن:

```text
model = Constructor(...)
model.fit(X_train, y_train)
pred = model.predict(X_test)
metric(...)
```

برای preprocessing:

```text
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)
```

برای خیلی از مدل‌ها همین pattern تکرار می‌شود.


In [ ]:

# A tiny sklearn API drill

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, mean_absolute_error

# Regression
lr_model = LinearRegression()
lr_model.fit(X, y)
print("LinearRegression:", lr_model.predict(X))

# Classification
log_model = LogisticRegression()
log_model.fit(X_cls, y_cls)
pred = log_model.predict(X_cls)

print("accuracy =", accuracy_score(y_cls, pred))
print("precision =", precision_score(y_cls, pred))
print("recall =", recall_score(y_cls, pred))
print("f1 =", f1_score(y_cls, pred))

# Others: remember constructor -> fit -> predict
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)
knn = KNeighborsClassifier(n_neighbors=3)
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
rf = RandomForestClassifier(n_estimators=50, random_state=42)
ada = AdaBoostClassifier(n_estimators=50, random_state=42)
gb = GradientBoostingClassifier(n_estimators=50, random_state=42)

kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
pca_model = PCA(n_components=2)

print("sklearn API drill finished.")



# 8. Final Exam Cheat Sheet

## Regression

```text
Linear:
y = Xw + b
MSE
gradient descent

Polynomial:
make x, x², x³...
then Linear Regression

Ridge:
MSE + λ||w||²

Lasso:
MSE + λΣ|w|
```

## Classification

```text
Logistic:
Xw+b → sigmoid → probability → threshold

Perceptron:
hard threshold → if wrong, update

KNN:
distance → nearest k → vote

Naive Bayes:
Bayes + conditional independence

SVM:
margin + hinge loss
```

## Trees

```text
Decision Tree:
best split
↓
recursive split
↓
leaf
```

## Ensembles

```text
Bagging:
bootstrap → independent models → vote/average

Random Forest:
bagging + random feature subsets

AdaBoost:
wrong samples get higher weight

Gradient Boosting:
next model learns residual / negative gradient
```

## Unsupervised

```text
K-Means:
assign → mean → repeat

GMM:
soft assignment + EM

PCA:
center → covariance/eigenvectors
or
center → SVD → projection
```

# 9. چیزهایی که باید قبل از خواب بتوانی از حفظ بنویسی

1. `X @ w + b`
2. Linear Regression gradient
3. Logistic + sigmoid + gradient
4. KNN: distance → sort → vote
5. Gini
6. Bagging vs Random Forest
7. AdaBoost weighted error + alpha
8. Gradient Boosting residual
9. K-Means assign/mean
10. PCA with SVD
11. train → fit → predict → metric
12. scaler: fit on train, transform on test



# 10. یک برگه‌ی فوق‌فشرده برای جلسه

\[
\boxed{Linear:\ \hat y=Xw+b}
\]

\[
\boxed{Logistic:\ p=\sigma(Xw+b)}
\]

\[
\boxed{Ridge:\ MSE+\lambda\|w\|^2}
\]

\[
\boxed{Lasso:\ MSE+\lambda\|w\|_1}
\]

\[
\boxed{Gini=1-\sum p_k^2}
\]

\[
\boxed{KMeans:\ Assign\rightarrow Mean}
\]

\[
\boxed{PCA:\ Center\rightarrow SVD\rightarrow Project}
\]

\[
\boxed{Bagging=Bootstrap+Parallel+Aggregate}
\]

\[
\boxed{RF=Bagging+Random\ Features}
\]

\[
\boxed{AdaBoost:\ Increase\ weights\ of\ mistakes}
\]

\[
\boxed{GB:\ Learn\ residuals/negative\ gradients}
\]

### مهم‌ترین mindset

> لازم نیست ۲۰ الگوریتم را با جزئیات حفظ کنی.  
> باید **الگوی اصلی هر الگوریتم + چند خط هسته‌ی implementation + API رایج** را بلد باشی.
